In [ ]:
# ============================================================
# Retail Intelligence Brasil
# Notebook.......: 14_bronze_sidra_populacao
# Camada.........: Bronze
# Fonte..........: IBGE - SIDRA
# Objetivo.......: Persistir a populacao residente da tabela SIDRA 4709 na Bronze.
# Autor..........: Cristhina Freire
# Data...........: 30/06/2026
# ============================================================



In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import requests
import json

from datetime import datetime



In [ ]:
# ============================================================
# 2. CONFIGURAÇÃO
# ============================================================

CATALOG = "retail_intelligence"
SCHEMA = "bronze"

TABLE_NAME = "ibge_sidra_populacao_raw"

FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

BASE_URL = "https://apisidra.ibge.gov.br"

SIDRA_TABLE_ID = 4709
SIDRA_VARIABLE_ID = 93
SIDRA_PERIOD = "2022"
SIDRA_LEVEL = "n6"
SIDRA_LOCATION = "all"

URL = (
    f"{BASE_URL}/values/"
    f"t/{SIDRA_TABLE_ID}/"
    f"{SIDRA_LEVEL}/{SIDRA_LOCATION}/"
    f"v/{SIDRA_VARIABLE_ID}/"
    f"p/{SIDRA_PERIOD}"
)

TIMEOUT = 60

CURRENT_DATE = datetime.now().strftime("%Y-%m-%d")
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/landing/"
    f"ibge/sidra/demografia/populacao/{CURRENT_DATE}"
)

FILE_NAME = f"populacao_{CURRENT_TIME}.json"

FILE_PATH = f"{VOLUME_PATH}/{FILE_NAME}"

print("=" * 60)
print("BRONZE - SIDRA - POPULAÇÃO RESIDENTE")
print("=" * 60)



In [ ]:
# ============================================================
# 3. EXTRAÇÃO
# ============================================================

print("Consumindo API SIDRA...")

response = requests.get(
    URL,
    timeout=TIMEOUT
)

response.raise_for_status()

sidra_response = response.json()

print(f"Registros retornados pela API: {len(sidra_response)}")



In [ ]:
# ============================================================
# 4. REMOVER CABEÇALHO
# ============================================================

header = sidra_response[0]

landing_data = sidra_response[1:]

print("Cabeçalho identificado.")

print(f"Registros válidos: {len(landing_data)}")



In [ ]:
# ============================================================
# 5. PERSISTÊNCIA DO JSON
# ============================================================

print("Salvando JSON na Landing...")

dbutils.fs.mkdirs(VOLUME_PATH)

dbutils.fs.put(
    FILE_PATH,
    json.dumps(
        landing_data,
        ensure_ascii=False,
        indent=2
    ),
    overwrite=True
)

print(f"Arquivo salvo em:\n{FILE_PATH}")



In [ ]:
# ============================================================
# 6. LEITURA DO JSON
# ============================================================

print("Lendo JSON salvo...")

landing_df = (
    spark.read
         .option("multiline", "true")
         .json(FILE_PATH)
)

print("Leitura concluída.")



In [ ]:
# ============================================================
# 7. PERSISTÊNCIA NA BRONZE
# ============================================================

print("Persistindo tabela Delta...")

(
    landing_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FULL_TABLE_NAME)
)

print("Tabela criada com sucesso.")



In [ ]:
# ============================================================
# 8. INSPEÇÃO
# ============================================================

bronze_df = spark.table(FULL_TABLE_NAME)

print("=" * 60)
print("INSPEÇÃO")
print("=" * 60)

print(f"Tabela...........: {FULL_TABLE_NAME}")
print(f"Total Registros..: {bronze_df.count()}")

bronze_df.printSchema()

display(bronze_df)



In [ ]:
# ============================================================
# 9. ENCERRAMENTO
# ============================================================

print("=" * 60)
print("Pipeline finalizado com sucesso.")
print("=" * 60)